# 02 - Preprocessing and RFM features

Cleaning funnel, RFM definitions, skewness and scaling (Phases 1 and 3).

This notebook reads the outputs in `results/` (produced by the scripts in `src/`); it does not repeat the cleaning logic.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").exists())
RES = ROOT / "results"
pd.set_option("display.width", 200)

def csv(name):
    path = RES / name
    if not path.exists():
        print(f"{name} not found: run `python src/run_all.py` from the repository root first.")
        return None
    return pd.read_csv(path)

def js(name):
    path = RES / name
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}


## Cleaning funnel

In [ ]:
cleaning = js('cleaning_summary.json')
retention = cleaning.get('cleaning_retention_percent')
if retention is None and cleaning.get('raw_rows'):
    retention = cleaning.get('clean_rows', 0) / cleaning['raw_rows'] * 100
print(f"{retention:.3f}% of raw rows retained" if retention is not None else "Cleaning retention unavailable")

## RFM definitions and statistics

In [ ]:
rfm = js('rfm_summary.json'); print(json.dumps(rfm.get('definitions', {}), indent=2)); print('reference date:', rfm.get('reference_date'))
display(csv('rfm_feature_statistics.csv')); display(csv('rfm_log_feature_statistics.csv'))

## Skewness before and after log1p

In [ ]:
raw = rfm.get('raw_feature_statistics', {})
if raw:
    skew = pd.DataFrame({'raw': {f: s['skewness'] for f, s in raw.items()}, 'log1p': rfm.get('log_feature_skewness', {})})
    display(skew); skew.plot(kind='bar', figsize=(6,3.5)); plt.ylabel('skewness'); plt.tight_layout(); plt.show()

## Distributions (needs `results/customer_rfm.csv`, generated locally)

In [ ]:
import numpy as np
cust = csv('customer_rfm.csv')
if cust is not None:
    fig, axes = plt.subplots(2, 3, figsize=(12, 6))
    for j, f in enumerate(['Recency','Frequency','Monetary']):
        axes[0, j].hist(cust[f], bins=50); axes[0, j].set_title(f'{f} (raw)')
        axes[1, j].hist(np.log1p(cust[f]), bins=50); axes[1, j].set_title(f'log1p({f})')
    plt.tight_layout(); plt.show()